In [0]:
%pip install pyyaml openpyxl requests

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

CODE_ROOT = Path(
    "/Workspace/Users/arfaashaikh101@gmail.com/retail_dev_dbx"
)

sys.path.insert(0, str(CODE_ROOT))

from scripts.bronze_loader import (
    load_config,
    setup,
    prepare_source,
    ingest_range,
    build_silver_and_gold,
    table,
)

config = load_config(
    spark,
    CODE_ROOT / "config/source_contracts.yaml",
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

# These new tables do not replace the earlier users/products/carts.
setup(spark, config, reset=False)

In [0]:
source_path, source_manifest = prepare_source(config)

print("Source:", source_path)
print("Original rows:", source_manifest["row_count"])

assert source_manifest["row_count"] == 1067371

In [0]:
run_id, status = ingest_range(
    spark,
    config,
    source_path,
    source_manifest,
    start="2009-12-01",
    end="2009-12-04",
    load_type="HISTORICAL",
)

print(run_id, status)

In [0]:
replay_id, replay_status = ingest_range(
    spark,
    config,
    source_path,
    source_manifest,
    start="2009-12-01",
    end="2009-12-04",
    load_type="HISTORICAL",
)

assert replay_status == "SKIPPED"

display(
    spark.table(table(config, "retail_monitoring"))
    .orderBy("started_at")
)

In [0]:
historical_run_id, historical_status = ingest_range(
    spark,
    config,
    source_path,
    source_manifest,
    start=config["source"]["start_date"],
    end=config["source"]["historical_end_exclusive"],
    load_type="HISTORICAL",
)

print(historical_run_id, historical_status)

In [0]:
from datetime import date, timedelta

start = date.fromisoformat(
    config["source"]["historical_end_exclusive"]
)
end = date.fromisoformat(
    config["source"]["end_date_exclusive"]
)

incremental_results = []
day = start

while day < end:
    next_day = day + timedelta(days=1)

    run_id, status = ingest_range(
        spark,
        config,
        source_path,
        source_manifest,
        start=day.isoformat(),
        end=next_day.isoformat(),
        load_type="INCREMENTAL_REPLAY",
    )

    incremental_results.append({
        "date": day.isoformat(),
        "run_id": run_id,
        "status": status,
    })

    day = next_day

display(spark.createDataFrame(incremental_results))

In [0]:
results = build_silver_and_gold(spark, config)
print(results)

In [0]:
bronze_count = spark.table(
    table(config, "bronze_transactions")
).count()

silver_count = spark.table(
    table(config, "silver_order_items")
).count()

print("Original:", source_manifest["row_count"])
print("Bronze:", bronze_count)
print("Silver:", silver_count)

assert bronze_count == source_manifest["row_count"]
assert silver_count == bronze_count

display(
    spark.table(table(config, "retail_monitoring"))
    .orderBy("started_at")
)

In [0]:
display(
    spark.table(table(config, "gold_daily_sales"))
    .orderBy("source_date", "country")
)

display(
    spark.table(table(config, "gold_product_sales"))
    .orderBy(F.col("net_signed_value_gbp").desc())
    .limit(20)
)